# لاب ٤ — العربية خطوة بخطوة
ابدأ بخلايا الإعداد بالترتيب. الخطوات الأولى لا تحتاج GPU أو ربط Drive. اختر Python 3.12؛ في Colab يمكن استخدام Runtime Version 2026.07. التدريب الاختياري يحتاج GPU وملفات تجربة لاب ٣.

التنفيذ والقياسات المحلية موثقة في `docs/LAB4_WALKTHROUGH.md`. البيانات لا تحتوي خليجيًا في التحقق، لذلك لا يمكن ادعاء تحسن Gulf منها.


In [ ]:
import sys, subprocess
RUNTIME_READY = REPO_READY = INSTALL_READY = False
assert sys.version_info[:2] == (3, 12), "اختر Runtime يدعم Python 3.12 ثم أعد الاتصال."
RUNTIME_READY = True
print("Python:", sys.version.split()[0])


## ١. جلب كود المشروع المحدّث
نستخدم مستودعك نفسه. التحديث يحافظ على التعديلات المحلية ويتوقف إذا تعارضت مع المصدر.


In [ ]:
import os, sys, subprocess
assert globals().get("RUNTIME_READY", False), "Complete the Python 3.12 / GPU setup cell first."
REPO_READY = INSTALL_READY = LAB3_TESTS_PASSED = False
from pathlib import Path
REPO_URL = "https://github.com/jnjnmaizi/BAYAN.DAICO.git"
PROJECT = Path("/content/BAYAN.DAICO")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    assert (PROJECT / ".git").is_dir(), "Existing project folder is not a Git checkout; preserve it before using a fresh runtime."
    remote = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Existing folder points to a different repository."
    branch = subprocess.check_output(["git", "-C", str(PROJECT), "branch", "--show-current"], text=True).strip()
    assert branch == "main", "Preserve your branch changes before switching to the course main branch."
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "merge", "--ff-only", "FETCH_HEAD"], check=True)
os.chdir(PROJECT)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))
assert (PROJECT / "src/bayan/models/training.py").exists(), "Lab 3 code is missing from this checkout. Check the repository version shown above."
REPO_READY = True


## ٢. تثبيت المكتبات
انتظر ظهور ALL GOOD. لا تتجاوز خطأ تثبيت قبل المتابعة.


In [ ]:
import os, sys, subprocess
INSTALL_READY = LAB3_TESTS_PASSED = False
assert globals().get("RUNTIME_READY", False) and globals().get("REPO_READY", False), "Complete runtime and repository setup before installing."
# This project uses PyTorch; do not load Colab's unrelated TensorFlow/Keras stack.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "scripts/doctor.py"], cwd=PROJECT, check=True)
# The running notebook kernel may not see a newly installed editable package yet.
source_dir = PROJECT / "src"
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))
import bayan
print("Bayan loaded from:", bayan.__file__)
INSTALL_READY = True


## ٣. تنزيل بيانات المحلل الصرفي
وجود camel-tools وحده لا يكفي؛ التحليل يحتاج قاعدة بيانات ونموذج MLE.


In [ ]:
import os, sys, subprocess
from pathlib import Path
assert globals().get("INSTALL_READY", False), "شغّل التثبيت أولًا."
COLAB_RESULTS = PROJECT / "artifacts/lab4_colab"
COLAB_RESULTS.mkdir(parents=True, exist_ok=True)
def run_lab4(*args):
    result = subprocess.run([sys.executable, "-u", *map(str,args)], cwd=PROJECT,
        env={**os.environ,"USE_TF":"0","USE_FLAX":"0"},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError("توقف الأمر؛ رسالة السبب كاملة أعلاه.")
run_lab4("scripts/setup_lab4.py")


## ٤. جرّب الفرق بين ملفّي التطبيع
لاحظ أن نص العرض يحتفظ بالهمزة والتشكيل، وأن الهاتف مخفي في النسختين.


In [ ]:
import sys
from pathlib import Path
source_dir = Path("/content/BAYAN.DAICO/src")
if not (source_dir / "bayan/preprocessing/arabic.py").is_file():
    raise RuntimeError("Project files are missing. Run the repository setup and installation cells first.")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from bayan.preprocessing.arabic import prepare_arabic_text, BAYAN_AR_V1, CAMELBERT_V1
for profile in [BAYAN_AR_V1, CAMELBERT_V1]:
    print(prepare_arabic_text("إضاءةُ مـدرسة 0551234567", profile))


## ٥. اختبارات الصحة
الاختبارات تفحص القواعد والمحاذاة وحفظ الأوزان؛ لا تدرّب نموذجًا كبيرًا.


In [ ]:
run_lab4("-m", "pytest", "tests/test_arabic_normalize.py", "tests/test_lab4_regressions.py", "-q")


## ٦. تدقيق اللهجات
نعدّ التسميات المرفقة فقط؛ راقب الفرق بين النسبة الإجمالية وتوزيع كل split.


In [ ]:
run_lab4("scripts/dialect_audit.py", "--output", COLAB_RESULTS / "dialect_audit.json")


## ٧. تقسيم اللواصق والمحاذاة
الجزء الأساسي في «وبالرياض» هو «رياض»؛ نستخدم موضعه لإرجاع تنبؤ NER للكلمة الأصلية.


In [ ]:
import sys
from pathlib import Path
source_dir = Path("/content/BAYAN.DAICO/src")
if not (source_dir / "bayan/preprocessing/arabic.py").is_file():
    raise RuntimeError("Project files are missing. Run the repository setup and installation cells first.")
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from bayan.preprocessing.arabic import segment, ner_segmented_view
print(segment("انقطعت الكهرباء وبالرياض تأخرت الصيانة"))
pieces, anchors = ner_segmented_view(["وبالرياض", "مرجعه", "BYN-000001"])
print("الأجزاء:", pieces)
print("الأجزاء الأساسية:", [pieces[i] for i in anchors])


## ٨. مراجعة النتائج المحلية المحفوظة
هذه الخلية تقرأ تقرير تشغيل الماك الموجود في GitHub؛ ليست تجربة Colab جديدة. نتيجة LOCATION انخفضت مع إدخال D3 إلى النموذج القديم، لذلك احتفظنا بالمسار الأصلي.


In [ ]:
import json
report = json.loads((PROJECT / "artifacts/lab4/ner_segmentation.json").read_text())
print("مصدر النتيجة: التشغيل المحلي المحفوظ في المستودع")
print("قبل:", report["baseline"]["LOCATION"])
print("بعد:", report["segmented"]["LOCATION"])
print("فرق recall بالنقاط:", report["location_recall_delta_points"])


## ٩. إعادة تجربة NER في Colab — عند توفر نموذج لاب ٣ في Drive
الخطوات السابقة تكفي لتنفيذ وفهم التطبيع والتقسيم ومراجعة الدليل. لإعادة قياس NER هنا، فعّل الخيار التالي فقط إذا كانت ملفات نموذج لاب ٣ و`metrics.json` و`validation_predictions.json` محفوظة في المسار المحدد. أوزان الماك الكبيرة ليست مرفوعة إلى GitHub.
فشل ربط Drive لا يمنع الخطوات ١–٨. نستخدم مجلد نتائج مستقلًا حتى لا ننسب نتائج الماك إلى Colab.


In [ ]:
RUN_NER_EVALUATION = False
if RUN_NER_EVALUATION:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB3_DIR = Path("/content/drive/MyDrive/BAYAN.DAICO/lab3_colab_01")
    model_dir = LAB3_DIR / "ner"
    required = ["model.safetensors", "metrics.json", "validation_predictions.json"]
    missing = [name for name in required if not (model_dir / name).is_file()]
    assert not missing, f"ملفات لاب ٣ غير متوفرة في {model_dir}: {missing}"
    output_dir = LAB3_DIR.parent / "lab4_colab_01"
    run_lab4("scripts/ner_segmentation_eval.py", "--model-dir", model_dir, "--output-dir", output_dir)
else:
    print("لم نعد تقييم NER في Colab؛ راجع النتائج المحلية في الخلية السابقة.")


## المقارنة الاختيارية
نتائج CAMeLBERT-Mix وDA المحلية موجودة في `artifacts/lab4/arabic_bakeoff.json`. لإعادة تدريب المقارنة على GPU استخدم `scripts/arabic_bakeoff.py --train` وحدد `--model-root` و`--output-dir` في Drive و`--incumbent-dir` لمسار مصنّف لاب ٣ المحفوظ. لا توجد أمثلة Gulf في التحقق الحالي؛ لا يمكن قياس هدف +4 نقاط على Gulf منه.

شرح القرارات الكامل: `docs/LAB4_WALKTHROUGH.md`.
